# Test: Live Cerebus NSP connection

Connects to a real Blackrock Cerebus NSP (or Central), runs the full
pipeline for a fixed duration, and reports detected events.

**Requirements:**
- NSP powered on and connected to the network
- `pip install direct-neural-biasing[live]`

Edit `INST_ADDR` and `N_CHANNELS` below to match your setup.

In [ ]:
# === Configuration — edit these for your rig ===
INST_ADDR = ""           # NSP IP, or "" for auto-discover
CLIENT_ADDR = "0.0.0.0"  # local interface to bind
N_CHANNELS = 83          # number of continuous channels
SAMPLE_RATE = 30000.0
RUN_SECONDS = 30         # how long to acquire

In [ ]:
import logging
import time

from dnb import CerebusSource, Pipeline, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, EventDetector, PowerEstimator
from dnb.io import ResultWriter

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")

config = PipelineConfig(
    sample_rate=SAMPLE_RATE,
    n_channels=N_CHANNELS,
    chunk_duration=0.2,
    buffer_duration=10.0,
)

writer = ResultWriter(output_dir="./live_output", prefix="cerebus")

pipeline = Pipeline(
    source=CerebusSource(inst_addr=INST_ADDR, client_addr=CLIENT_ADDR),
    modules=[
        WaveletConvolution(freq_min=1, freq_max=200, n_freqs=40),
        PowerEstimator(),
        EventDetector(
            event_type=EventType.RIPPLE,
            freq_range=(80, 250),
            threshold_std=3.0,
            warmup_chunks=10,
        ),
    ],
    config=config,
)

# Log every event and accumulate for saving
event_count = 0

def on_event(event):
    global event_count
    event_count += 1
    writer.on_event(event)
    if event_count <= 20:
        print(
            f"  [{event_count:3d}] {event.event_type.name} "
            f"ch={event.channel_id} t={event.timestamp:.3f}s "
            f"dur={event.duration:.3f}s "
            f"peak={event.metadata.get('peak_amplitude', 0):.2f}"
        )

pipeline.on_event(None, on_event)  # subscribe to all event types

# Run for a fixed duration then stop
print(f"Starting live acquisition for {RUN_SECONDS}s...")
print(f"  NSP: {INST_ADDR or 'auto-discover'}")
print(f"  Channels: {N_CHANNELS}, Sample rate: {SAMPLE_RATE:.0f} Hz")
print()

import threading
timer = threading.Timer(RUN_SECONDS, pipeline.stop)
timer.start()

try:
    pipeline.run_online()
except KeyboardInterrupt:
    pipeline.stop()
finally:
    timer.cancel()

print(f"\nDone. Detected {event_count} events in {RUN_SECONDS}s.")

In [ ]:
# Save results
events_path = writer.save_events()
print(f"Events saved to {events_path}")